### 🎯 [미션] 주석이 달린 줄의 None 또는 _______ 부분을 채워 코드를 완성하세요.

각 셀을 순서대로 실행(Shift + Enter)해야 합니다.

---

# 🌱 식물 질병 진단 시스템

이 노트북에서는 학습된 모델을 사용하여 **실제 식물 질병 진단 시스템**을 구현합니다.

**시스템 기능:**
1. 토마토 잎 이미지를 입력받아 질병 진단
2. 진단 결과와 신뢰도 표시
3. 질병별 대처 방법 안내

---
#### 1️⃣ 필요한 라이브러리 불러오기

In [ ]:
# YOLO 라이브러리 설치 (최초 1회만 실행)
!pip install ultralytics

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
from PIL import Image as PILImage

from ultralytics import YOLO

print("✅ 라이브러리 로딩 완료!")

---
#### 2️⃣ 경로 설정 및 모델 로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd "/content/drive/MyDrive/2026_AI_Advanced_Study-main/4차시/03_plant_disease/code_GT"

In [ ]:
# 학습된 모델의 경로 설정
model_path = './runs/classify/train/weights/best.pt'

# 모델 로드
model = YOLO(model_path)

print(f"✅ 모델 로드 완료: {model_path}")

---
#### 3️⃣ 질병 정보 데이터베이스 구축

각 질병에 대한 설명과 대처 방법을 정의합니다.

In [ ]:
# 질병별 정보를 담은 딕셔너리
# 각 질병에 대한 한글 이름, 설명, 대처 방법이 정의되어 있습니다.

DISEASE_INFO = {
    'Tomato_healthy': {
        'name': '건강',
        'description': '건강한 토마토 잎입니다.',
        'treatment': '현재 상태를 유지하세요. 정기적인 관찰을 권장합니다.',
        'severity': 'none'
    },
    'Tomato_Bacterial_spot': {
        'name': '세균성 점무늬병',
        'description': '세균(Xanthomonas)에 의해 발생하는 질병으로, 잎에 작은 갈색 반점이 나타납니다.',
        'treatment': '감염된 잎을 제거하고, 구리 기반 살균제를 살포하세요. 물을 줄 때 잎에 직접 닿지 않도록 합니다.',
        'severity': 'medium'
    },
    'Tomato_Early_blight': {
        'name': '잎마름병 (겹무늬병)',
        'description': '곰팡이(Alternaria solani)에 의해 발생하며, 동심원 무늬의 갈색 반점이 특징입니다.',
        'treatment': '감염된 잎을 즉시 제거하고, 살균제를 살포하세요. 식물 간격을 넓혀 통풍을 개선합니다.',
        'severity': 'high'
    },
    'Tomato_Leaf_Mold': {
        'name': '잎곰팡이병',
        'description': '곰팡이(Passalora fulva)에 의해 발생하며, 잎 뒷면에 올리브색 곰팡이가 생깁니다.',
        'treatment': '온실 환기를 개선하고 습도를 낮추세요. 감염된 잎을 제거하고 살균제를 사용합니다.',
        'severity': 'medium'
    },
    'Tomato_Yellow_Leaf_Curl_Virus': {
        'name': '황화잎말림바이러스 (TYLCV)',
        'description': '담배가루이가 전파하는 바이러스 질병으로, 잎이 노랗게 변하고 말립니다.',
        'treatment': '감염된 식물을 격리 또는 제거하세요. 담배가루이 방제가 중요합니다. 저항성 품종 재배를 권장합니다.',
        'severity': 'critical'
    }
}

# 심각도별 색상 정의
SEVERITY_COLORS = {
    'none': '#28a745',      # 초록색 - 건강
    'medium': '#ffc107',    # 노란색 - 중간
    'high': '#fd7e14',      # 주황색 - 높음
    'critical': '#dc3545'   # 빨간색 - 심각
}

print("✅ 질병 정보 데이터베이스 로드 완료!")
print(f"   등록된 질병 수: {len(DISEASE_INFO)}개")

---
#### 4️⃣ 진단 함수 구현

이미지를 입력받아 질병을 진단하는 함수를 구현합니다.

In [ ]:
def diagnose_plant(image_path):
    """
    토마토 잎 이미지를 분석하여 질병을 진단합니다.
    
    Args:
        image_path: 진단할 이미지 경로
        
    Returns:
        dict: 진단 결과 (클래스, 신뢰도, 질병 정보)
    """
    # 🎯 [미션] model의 예측 메서드를 호출하세요.
    # 힌트: 학습은 train(), 검증은 val(), 예측은?
    results = model.predict(image_path, verbose=False)
    
    # 🎯 [미션] 결과에서 확률 정보를 추출하세요.
    # 힌트: Classification 결과에서 확률 객체의 이름은?
    probs = results[0].probs
    
    # 🎯 [미션] 가장 높은 확률의 클래스 인덱스를 가져오세요.
    # 힌트: 상위 1개 클래스의 인덱스를 반환하는 속성은?
    predicted_idx = probs.top1
    predicted_class = results[0].names[predicted_idx]
    confidence = probs.top1conf.item()
    
    # 질병 정보 가져오기
    disease_info = DISEASE_INFO.get(predicted_class, {
        'name': '알 수 없음',
        'description': '등록되지 않은 클래스입니다.',
        'treatment': '-',
        'severity': 'none'
    })
    
    return {
        'class': predicted_class,
        'confidence': confidence,
        'info': disease_info
    }

---
#### 5️⃣ 진단 결과 시각화 함수

In [ ]:
def visualize_diagnosis(image_path, diagnosis_result):
    """
    진단 결과를 시각화합니다.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 왼쪽: 입력 이미지
    img = PILImage.open(image_path)
    axes[0].imshow(img)
    axes[0].set_title('Input Image', fontsize=14)
    axes[0].axis('off')
    
    # 오른쪽: 진단 결과
    info = diagnosis_result['info']
    severity_color = SEVERITY_COLORS.get(info['severity'], '#6c757d')
    
    # 텍스트 박스 생성
    axes[1].set_xlim(0, 10)
    axes[1].set_ylim(0, 10)
    axes[1].axis('off')
    
    # 제목
    axes[1].text(5, 9.5, '🌱 진단 결과', fontsize=18, ha='center', fontweight='bold')
    
    # 진단명
    axes[1].text(0.5, 8, f"진단: {info['name']}", fontsize=16, color=severity_color, fontweight='bold')
    
    # 신뢰도
    confidence_pct = diagnosis_result['confidence'] * 100
    axes[1].text(0.5, 7, f"신뢰도: {confidence_pct:.1f}%", fontsize=14)
    
    # 설명
    axes[1].text(0.5, 5.5, "📋 설명:", fontsize=12, fontweight='bold')
    # 긴 텍스트 줄바꿈
    desc = info['description']
    if len(desc) > 40:
        desc = desc[:40] + '\n' + desc[40:]
    axes[1].text(0.5, 4.5, desc, fontsize=11, wrap=True)
    
    # 대처 방법
    axes[1].text(0.5, 3, "💊 대처 방법:", fontsize=12, fontweight='bold')
    treatment = info['treatment']
    if len(treatment) > 40:
        treatment = treatment[:40] + '\n' + treatment[40:80] + '\n' + treatment[80:]
    axes[1].text(0.5, 1.5, treatment, fontsize=11, wrap=True)
    
    plt.tight_layout()
    plt.show()

---
#### 6️⃣ 시스템 테스트

실제 이미지로 진단 시스템을 테스트합니다.

In [ ]:
import glob

# 테스트 이미지 수집 (각 클래스에서 1장씩)
test_path = '../data/test'
test_images = []

for cls_folder in sorted(os.listdir(test_path)):
    cls_path = os.path.join(test_path, cls_folder)
    if os.path.isdir(cls_path):
        images = glob.glob(os.path.join(cls_path, '*.[jJ][pP][gG]'))
        if images:
            test_images.append(images[0])

print(f"📷 테스트할 이미지: {len(test_images)}장")

In [ ]:
# 각 이미지에 대해 진단 수행
for img_path in test_images:
    print("\n" + "=" * 60)
    print(f"📁 파일: {os.path.basename(img_path)}")
    print(f"📂 실제 클래스: {os.path.basename(os.path.dirname(img_path))}")
    print("=" * 60)
    
    # 진단 수행
    result = diagnose_plant(img_path)
    
    # 결과 시각화
    visualize_diagnosis(img_path, result)